# 🔍 UYDS SSS Veri Ayıklama Pipeline v2.1
## A100 GPU · Tek Model · 12 Hata Kategorisi · Eşik Kalibrasyonu

**v2.1 düzeltmeleri (v2'den):**
- 🔴 `score_function` string → import `dot_score` / `cos_sim`
- 🔴 Tam tekrar mantığı düzeltildi (set kaldırıldı)
- 🔴 Doğrulama seti üzerine yazma koruması
- 🟠 Madde no embedding'den çıkarıldı (semantik bozulma yok)
- 🟠 Leave-one-out centroid (kendini dışarıda bırak)
- 🟠 Rastgele örneklem (tek kategoriye kayma yok)
- 🟠 2 çıktı: `korunan_veri.xlsx` + `otomatik_silinecekler.xlsx`
- 🟢 FAISS açıklaması düzeltildi
- 🟢 `torch` pip listesinden çıkarıldı (Colab uyumluluğu)
- 🟢 Colab / VS Code ortam kontrolü

**Çıktılar:**
- `korunan_veri.xlsx` — OTOMATİK_SİL olmayan tüm satırlar (İNCELE + GÜÇLÜ_ADAY + TEMİZ)
- `otomatik_silinecekler.xlsx` — Sadece OTOMATİK_SİL satırlar
- `tum_kayitlar_raporu.xlsx` — Tüm satırlar + hata sınıfları + eşleşen kayıt
- `ozet.json` — İstatistiksel özet
---

## ⚙️ Hücre 0: Ortam Kurulumu (Colab + VS Code uyumlu)

In [ ]:
# ============================================================
# HÜCRE 0: Ortam kurulumu
# ============================================================
import sys, os, warnings
warnings.filterwarnings("ignore")

# --- Colab / VS Code tespiti ---
COLAB_MI = "google.colab" in sys.modules
print(f"🖥️ Ortam: {'Google Colab' if COLAB_MI else 'Yerel VS Code / Jupyter'}")

if COLAB_MI:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    # GPU kontrolü (Colab'da)
    gpu_info = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null
    if gpu_info:
        print(f"✅ GPU: {gpu_info[0]}")
    else:
        print("⚠️ GPU bulunamadı. CPU ile devam ediliyor (yavaş olabilir).")

    # Pip kurulumları (torch YOK — Colab'ın mevcut sürümü kullanılır)
    print("📦 Paketler yükleniyor...")
    !pip install -qU \
        sentence-transformers==3.4.1 \
        transformers==4.47.1 \
        pandas==2.2.3 \
        openpyxl==3.1.5 \
        scikit-learn==1.6.0 \
        lingua-language-detector==2.0.2 \
        hunspell==0.5.5 \
        tqdm==4.67.1
    print("✅ Paketler yüklendi.")

    # Hunspell Türkçe sözlük
    if not os.path.exists("/usr/share/hunspell/tr_TR.dic"):
        !apt-get install -y hunspell-tr > /dev/null 2>&1
        print("✅ Hunspell Türkçe sözlük kuruldu.")
else:
    print("ℹ️ Yerel ortam: pip paketlerini manuel kurduğunuzu varsayıyorum.")

# --- Çalışma dizini ---
import glob as _glob
if COLAB_MI:
    CALISMA_DIZINI = "/content/drive/MyDrive"
    adaylar = _glob.glob("/content/drive/MyDrive/**/google_collab_egitim", recursive=True)
    if adaylar:
        CALISMA_DIZINI = adaylar[0]
else:
    CALISMA_DIZINI = os.getcwd()

print(f"📁 Çalışma dizini: {CALISMA_DIZINI}")

# --- Excel dosyası ---
EXCEL_DOSYASI = os.path.join(CALISMA_DIZINI, "UYDS_SSS_Listesi.xlsx")
if not os.path.exists(EXCEL_DOSYASI):
    raise FileNotFoundError(
        f"❌ Excel dosyası bulunamadı: {EXCEL_DOSYASI}\n"
        f"   Lütfen '{os.path.basename(CALISMA_DIZINI)}' klasörüne dosyayı koyun."
    )

CIKTI_DIZINI = os.path.join(CALISMA_DIZINI, "ciktilar")
os.makedirs(CIKTI_DIZINI, exist_ok=True)

print(f"📄 Excel: {EXCEL_DOSYASI}")
print(f"📁 Çıktı: {CIKTI_DIZINI}")
print("✅ Hücre 0 tamam.")

## 📖 Hücre 1: Excel'i Oku + Sütun Eşleştir

In [ ]:
# ============================================================
# HÜCRE 1: Oku + sütunları eşleştir
# ============================================================
import pandas as pd
import numpy as np
import json, re, unicodedata
from datetime import datetime

print(f"📖 Okunuyor: {EXCEL_DOSYASI}")
veri = pd.read_excel(EXCEL_DOSYASI, sheet_name=0, engine="openpyxl", dtype=object)
print(f"✅ {len(veri)} satır × {len(veri.columns)} sütun: {list(veri.columns)}")

def adi_normalize_et(ad: str) -> str:
    if not isinstance(ad, str): return ""
    ad = unicodedata.normalize("NFKC", ad)
    return ad.strip().lower().replace("ı","i").replace("ğ","g").replace("ü","u").replace("ş","s").replace("ö","o").replace("ç","c")

mevcut_kolonlar = {adi_normalize_et(k): k for k in veri.columns}
HEDEF_KOLONLAR = {
    "modul": ["modul"], "konu": ["konu"], "aciklama": ["aciklama"],
    "yapilacak": ["yapilacak"], "yapilan_islem": ["yapilan islem", "yapilan_islem"],
    "madde_no": ["madde no", "madde_no", "harici kod"],
    "madde_tarihi": ["madde tarihi", "madde_tarihi"], "musteri": ["musteri"],
}
STANDART_KOLONLAR = {"kategori": ["kategori"], "soru": ["soru"], "cevap": ["cevap"],
    "kaynak": ["kaynak dokuman", "kaynak belge", "kaynak"]}

def kolonlari_esle(hedef):
    esleme = {}
    for alan, adaylar in hedef.items():
        for aday in adaylar:
            if adi_normalize_et(aday) in mevcut_kolonlar:
                esleme[alan] = mevcut_kolonlar[adi_normalize_et(aday)]; break
    return esleme

uyds_esleme = kolonlari_esle(HEDEF_KOLONLAR)
standart_esleme = kolonlari_esle(STANDART_KOLONLAR)
format_turu = "uyds" if len(uyds_esleme) >= 4 else ("standart" if len(standart_esleme) >= 3 else None)
esleme = uyds_esleme if format_turu == "uyds" else standart_esleme
if format_turu is None:
    raise ValueError(f"Excel formatı tanınamadı: {list(mevcut_kolonlar.keys())}")
print(f"✅ Format: {format_turu}. Eşlenen: {list(esleme.keys())}")

def metne_donustur(deger) -> str:
    if deger is None or (isinstance(deger, float) and pd.isna(deger)): return ""
    if isinstance(deger, (int, float)): return str(int(deger)) if deger == int(deger) else str(deger)
    return str(deger).strip()

veri.head(2)

## 📝 Hücre 2: Kayıt Metni Oluştur (Birleşik — Madde No Embedding'de YOK)

**v2.1:** Madde numarası embedding metnine eklenmez (semantik benzerliği bozmaması için).
Raporda ayrı sütun olarak kalır.

In [ ]:
# ============================================================
# HÜCRE 2: Kayıt metni oluştur (madde no embedding'de YOK)
# ============================================================

if format_turu == "uyds":
    for alan in ["modul", "konu", "aciklama", "yapilacak", "yapilan_islem", "madde_no", "madde_tarihi", "musteri"]:
        if alan in esleme:
            veri[f"_{alan}"] = veri[esleme[alan]].apply(metne_donustur)
        else:
            veri[f"_{alan}"] = ""
else:
    veri["_kategori_1"] = veri[esleme["kategori"]].apply(metne_donustur)
    veri["_soru_1"] = veri[esleme["soru"]].apply(metne_donustur)
    veri["_cevap_1"] = veri[esleme["cevap"]].apply(metne_donustur)
    veri["_modul"] = veri["_kategori_1"]
    veri["_madde_no"] = ""

# --- Birleşik kayıt metni (madde_no YOK) ---
def kayit_metni_olustur(satir):
    parcalar = []
    if format_turu == "uyds":
        if satir.get("_modul"): parcalar.append(f"[{satir['_modul']}]")
        if satir.get("_konu"): parcalar.append(satir["_konu"])
        if satir.get("_aciklama"): parcalar.append(satir["_aciklama"])
        if satir.get("_yapilacak"): parcalar.append(f"Yapılacak: {satir['_yapilacak']}")
        if satir.get("_yapilan_islem"): parcalar.append(f"Yapılan İşlem: {satir['_yapilan_islem']}")
        # Madde no embedding'e EKLENMEZ (semantik bozulmayı önler)
    else:
        if satir.get("_kategori_1"): parcalar.append(f"[{satir['_kategori_1']}]")
        if satir.get("_soru_1"): parcalar.append(satir["_soru_1"])
        if satir.get("_cevap_1"): parcalar.append(satir["_cevap_1"])
    return ". ".join(parcalar) if parcalar else ""

veri["_kayit_metni"] = veri.apply(kayit_metni_olustur, axis=1)
veri["_kategori"] = veri["_modul"]

bos_metin = (veri["_kayit_metni"] == "").sum()
print(f"📊 {len(veri)} satır, {veri['_kategori'].nunique()} benzersiz kategori, {bos_metin} boş")
print("📋 İlk 2 kayıt metni:")
for i, m in enumerate(veri["_kayit_metni"].head(2)):
    print(f"   [{i}] {m[:200]}")
print("✅ Hücre 2 tamam.")

## 📊 Hücre 3: Doğrulama Seti Oluştur (Rastgele Örneklem + Üzerine Yazma Koruması)

In [ ]:
# ============================================================
# HÜCRE 3: Doğrulama seti (rastgele, üzerine yazma YOK)
# ============================================================

DOGRULAMA_BOYUTU = min(200, len(veri))
veri["_manuel_etiket"] = ""

# Rastgele örneklem (tek kategoriye kaymayı önler)
dogrulama_indeksleri = veri.sample(n=DOGRULAMA_BOYUTU, random_state=42).index
veri.loc[dogrulama_indeksleri, "_manuel_etiket"] = "bekliyor"

dogrulama_seti = veri.loc[dogrulama_indeksleri,
    ["_modul", "_konu", "_kayit_metni", "_madde_no", "_manuel_etiket"]
].copy()

dogrulama_yolu = os.path.join(CIKTI_DIZINI, "dogrulama_seti.xlsx")

# ÜZERİNE YAZMA KORUMASI
if os.path.exists(dogrulama_yolu):
    print(f"⚠️ Mevcut doğrulama seti korundu: {dogrulama_yolu}")
    print("   Üzerine yazılmadı. Yeniden oluşturmak için dosyayı silin.")
else:
    dogrulama_seti.to_excel(dogrulama_yolu, index=False, engine="openpyxl")
    print(f"📋 Doğrulama seti oluşturuldu: {dogrulama_yolu} ({DOGRULAMA_BOYUTU} satır)")
    print("\n⚠️ ÖNEMLİ: Dosyayı indirip '_manuel_etiket' sütununu doldurun:")
    print("   'temiz' / 'hatali' / 'sil'")

print(f"✅ Hücre 3 tamam.")

## 🔍 Hücre 4: Yapısal Hata Tespiti

In [ ]:
# ============================================================
# HÜCRE 4: Yapısal kontroller
# ============================================================

SADECE_SEMBOL_RE = re.compile(r"^[\d\W_]+$")

def token_sayisi(m): return len(str(m).split()) if m else 0
def sadece_sembol_mu(m): return bool(SADECE_SEMBOL_RE.match(str(m))) if m else False
def kelime_cesitliligi(m):
    k = str(m).lower().split(); return len(set(k))/len(k) if k else 0.0

KISA_METIN_ESIK, UZUN_METIN_ESIK = 5, 300
hata_kolonlari = []

veri["_token_sayisi"] = veri["_kayit_metni"].apply(token_sayisi)
veri["_hata_bos"] = veri["_kayit_metni"] == ""; hata_kolonlari.append("_hata_bos")
veri["_hata_kisa"] = (veri["_token_sayisi"] > 0) & (veri["_token_sayisi"] < KISA_METIN_ESIK); hata_kolonlari.append("_hata_kisa")
veri["_hata_uzun"] = veri["_token_sayisi"] > UZUN_METIN_ESIK; hata_kolonlari.append("_hata_uzun")
veri["_hata_sembol"] = veri["_kayit_metni"].apply(sadece_sembol_mu); hata_kolonlari.append("_hata_sembol")
veri["_ttr"] = veri["_kayit_metni"].apply(kelime_cesitliligi)
veri["_hata_dusuk_ttr"] = (veri["_token_sayisi"] > 10) & (veri["_ttr"] < 0.25); hata_kolonlari.append("_hata_dusuk_ttr")

print("🔍 Yapısal kontrol:")
for k in hata_kolonlari:
    s = veri[k].sum()
    if s > 0: print(f"   {k}: {s}")
print("✅ Hücre 4 tamam.")

## 🌐 Hücre 5: Dil Tespiti (min 8 kelime)

In [ ]:
# ============================================================
# HÜCRE 5: Dil tespiti
# ============================================================
from lingua import Language, LanguageDetectorBuilder
from tqdm import tqdm

DIL_MIN_KELIME = 8
languages = [Language.TURKISH, Language.ENGLISH, Language.ARABIC, Language.RUSSIAN, Language.FRENCH, Language.GERMAN]
detector = LanguageDetectorBuilder.from_languages(*languages).build()

def dili_tespit_et(metin, ts):
    if not metin or not str(metin).strip(): return "bos"
    if ts < DIL_MIN_KELIME: return "belirsiz"
    try:
        s = detector.detect_language_of(str(metin)[:300])
        return s.iso_code_639_1.name.lower() if s else "belirsiz"
    except: return "belirsiz"

print("🌐 Dil tespiti...")
veri["_dil"] = veri.apply(lambda r: dili_tespit_et(r["_kayit_metni"], r["_token_sayisi"]), axis=1)
veri["_hata_dil"] = ~veri["_dil"].isin(["turkish", "bos", "belirsiz"])
hata_kolonlari.append("_hata_dil")
print(f"📊 Dil dağılımı:\n{veri['_dil'].value_counts()}")
print("✅ Hücre 5 tamam.")

## 📝 Hücre 6: Yazım Hatası Tespiti (%35, min 10 kelime)

In [ ]:
# ============================================================
# HÜCRE 6: Yazım denetimi
# ============================================================
import hunspell

YAZIM_ESIK, YAZIM_MIN_KELIME = 0.35, 10
hobj = None
for yol in ["/usr/share/hunspell/tr_TR", "/usr/share/myspell/tr_TR"]:
    try: hobj = hunspell.HunSpell(yol+".dic", yol+".aff"); print(f"✅ Hunspell yüklendi: {yol}"); break
    except: pass

def yazim_orani(metin):
    if not metin or hobj is None: return 0.0
    kelimeler = re.findall(r"\b[a-zA-ZçğıöşüÇĞİÖŞÜ]{3,}\b", str(metin).lower())
    if len(kelimeler) < YAZIM_MIN_KELIME: return 0.0
    return sum(1 for k in kelimeler if not hobj.spell(k)) / len(kelimeler)

veri["_yazim_orani"] = veri["_kayit_metni"].apply(yazim_orani)
veri["_hata_yazim"] = veri["_yazim_orani"] > YAZIM_ESIK
hata_kolonlari.append("_hata_yazim")
print(f"📝 Yazım hatası: {veri['_hata_yazim'].sum()} satır")
print("✅ Hücre 6 tamam.")

## 🧠 Hücre 7: Embedding Modelini Yükle

In [ ]:
# ============================================================
# HÜCRE 7: Model yükle
# ============================================================
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ Cihaz: {device}")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

print("📥 paraphrase-multilingual-mpnet-base-v2 (768d)...")
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2", device=device, trust_remote_code=False)
if device == "cuda":
    model.half(); print("   ✅ FP16")
boyut = model.get_sentence_embedding_dimension()
print(f"✅ Model hazır. Boyut: {boyut}")

## 🔢 Hücre 8: Embedding Üret (Numpy Array)

In [ ]:
# ============================================================
# HÜCRE 8: Embedding üret
# ============================================================
import time, gc

gecerli_mask = veri["_kayit_metni"].str.strip() != ""
metinler = veri.loc[gecerli_mask, "_kayit_metni"].tolist()
gecerli_indeksler = veri[gecerli_mask].index.tolist()

print(f"🔢 Embedding: {len(metinler)} satır")
t0 = time.time()
tum_vektorler = model.encode(metinler, batch_size=64, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
print(f"   {tum_vektorler.shape}, {time.time()-t0:.1f}s, {tum_vektorler.nbytes/1e6:.1f} MB")

veri["_vektor_indeksi"] = -1
for i, idx in enumerate(gecerli_indeksler):
    veri.at[idx, "_vektor_indeksi"] = i

print("✅ Hücre 8 tamam.")

## 🔁 Hücre 9: Semantic Search (Top-K, cosine similarity)

`sentence_transformers.util.semantic_search` kullanılır. Tam n×n matris tutulmaz, sorgu ve corpus arasında parçalı cosine hesaplanır.

In [ ]:
# ============================================================
# HÜCRE 9: Semantic search ile benzer kayıt
# ============================================================
from sentence_transformers.util import semantic_search, dot_score

BENZERLIK_GUCLU_ADAY = 0.95
BENZERLIK_INCELE = 0.88
TOP_K = 10

n = len(tum_vektorler)
print(f"🔍 Semantic search: {n} kayıt, top_k={TOP_K}...")

t0 = time.time()
# dot_score kullan (embedding'ler zaten normalize, cosine = dot product)
arama_sonuclari = semantic_search(tum_vektorler, tum_vektorler, top_k=TOP_K + 1, score_function=dot_score)
print(f"   Süre: {time.time()-t0:.1f}s")

veri["_benzer_indeksi"] = -1
veri["_benzer_skor"] = 0.0
veri["_benzer_madde_no"] = ""
veri["_benzer_metin"] = ""
veri["_benzer_seviye"] = "temiz"

guclu_aday = 0
incele = 0
tam_tekrar = 0

for i, sonuclar in enumerate(arama_sonuclari):
    idx = gecerli_indeksler[i]
    mevcut_metin = str(veri.at[idx, "_kayit_metni"])

    en_yuksek_skor = -1.0
    en_yuksek_j = -1

    for eslesme in sonuclar:
        j = eslesme["corpus_id"]
        skor = float(eslesme["score"])
        if j >= i: continue  # sadece önceki satırlar
        if skor > en_yuksek_skor:
            en_yuksek_skor = skor
            en_yuksek_j = j

    if en_yuksek_j >= 0 and en_yuksek_skor > 0:
        eslesen_idx = gecerli_indeksler[en_yuksek_j]
        eslesen_metin = str(veri.at[eslesen_idx, "_kayit_metni"])

        veri.at[idx, "_benzer_indeksi"] = eslesen_idx
        veri.at[idx, "_benzer_skor"] = en_yuksek_skor
        veri.at[idx, "_benzer_madde_no"] = str(veri.at[eslesen_idx, "_madde_no"])
        veri.at[idx, "_benzer_metin"] = eslesen_metin[:200]

        # Tam metin tekrarı: normalize edilmiş birebir eşleşme
        if mevcut_metin.strip().lower() == eslesen_metin.strip().lower():
            veri.at[idx, "_benzer_seviye"] = "tam_tekrar"
            tam_tekrar += 1
        elif en_yuksek_skor >= BENZERLIK_GUCLU_ADAY:
            veri.at[idx, "_benzer_seviye"] = "guclu_aday"
            guclu_aday += 1
        elif en_yuksek_skor >= BENZERLIK_INCELE:
            veri.at[idx, "_benzer_seviye"] = "incele"
            incele += 1

veri["_hata_tam_tekrar"] = veri["_benzer_seviye"] == "tam_tekrar"
veri["_hata_benzer_guclu"] = veri["_benzer_seviye"] == "guclu_aday"
veri["_hata_benzer_incele"] = veri["_benzer_seviye"] == "incele"
hata_kolonlari.extend(["_hata_tam_tekrar", "_hata_benzer_guclu", "_hata_benzer_incele"])

print(f"\n📊 Tam tekrar: {tam_tekrar} | Güçlü aday: {guclu_aday} | İncele: {incele}")

del arama_sonuclari; gc.collect()
if device == "cuda": torch.cuda.empty_cache()
print("✅ Hücre 9 tamam.")

## 🎯 Hücre 10: Outlier Tespiti (DBSCAN + IsoForest auto)

In [ ]:
# ============================================================
# HÜCRE 10: Outlier tespiti
# ============================================================
from sklearn.cluster import DBSCAN
from sklearn.ensemble import IsolationForest

print("🔍 DBSCAN...")
outlier_dbscan = DBSCAN(eps=0.35, min_samples=5, metric="cosine", n_jobs=-1).fit_predict(tum_vektorler) == -1
print(f"   DBSCAN outlier: {outlier_dbscan.sum()}")

# Dinamik contamination
if n < 500: cv = 0.02
elif n < 2000: cv = 0.03
elif n < 10000: cv = 0.04
else: cv = "auto"
print(f"🌲 Isolation Forest (contamination={cv})...")
outlier_iso = IsolationForest(contamination=cv, random_state=42, n_jobs=-1).fit_predict(tum_vektorler) == -1
print(f"   IsoForest outlier: {outlier_iso.sum()}")

veri["_hata_outlier"] = False
for i, idx in enumerate(gecerli_indeksler):
    if outlier_dbscan[i] or outlier_iso[i]:
        veri.at[idx, "_hata_outlier"] = True
hata_kolonlari.append("_hata_outlier")
print(f"📊 Toplam outlier: {veri['_hata_outlier'].sum()}")

gc.collect()
if device == "cuda": torch.cuda.empty_cache()
print("✅ Hücre 10 tamam.")

## 🏷️ Hücre 11: Kayıt ↔ Kategori Centroid (Leave-One-Out)

**v2.1:** Her kayıt kendi centroid hesabından çıkarılır (leave-one-out).

In [ ]:
# ============================================================
# HÜCRE 11: Kategori centroid (leave-one-out)
# ============================================================
from collections import defaultdict

KATEGORI_ESIK = 0.25
KATEGORI_MIN_KAYIT = 4  # leave-one-out için en az 4 (3 + kendisi)

kategori_vektorleri = defaultdict(list)
for i, idx in enumerate(gecerli_indeksler):
    kat = str(veri.at[idx, "_kategori"])
    if kat and kat not in ("nan", "Belirtilmemiş"):
        kategori_vektorleri[kat].append(i)

print(f"🏷️ Kategori: {len(kategori_vektorleri)} (≥{KATEGORI_MIN_KAYIT} kayıtlı olanlar centroid alır)")

veri["_kategori_cosine"] = -1.0
veri["_hata_kategori"] = False
uyumsuz = 0

for kat, v_indeksler in kategori_vektorleri.items():
    if len(v_indeksler) < KATEGORI_MIN_KAYIT:
        continue
    vektorler = tum_vektorler[v_indeksler]
    toplam = vektorler.sum(axis=0)
    n_kat = len(v_indeksler)

    for yerel_i, global_i in enumerate(v_indeksler):
        # Leave-one-out centroid: kendini çıkar
        centroid = (toplam - tum_vektorler[global_i]) / (n_kat - 1)
        centroid = centroid / (np.linalg.norm(centroid) + 1e-8)
        cos = float(np.dot(tum_vektorler[global_i], centroid))

        idx = gecerli_indeksler[global_i]
        veri.at[idx, "_kategori_cosine"] = cos
        if cos < KATEGORI_ESIK:
            veri.at[idx, "_hata_kategori"] = True
            uyumsuz += 1

hata_kolonlari.append("_hata_kategori")
print(f"📊 Kategori uyumsuz (LOO centroid): {uyumsuz}")
print("✅ Hücre 11 tamam.")

## 📋 Hücre 12: Hata Konsolidasyonu (4 Seviye)

- `OTOMATİK_SİL`: Boş, sadece sembol, tam metin tekrarı
- `GÜÇLÜ_ADAY`: Cosine > 0.95
- `İNCELE`: Diğer tüm şüpheli durumlar
- `TEMİZ`: Sorun yok

In [ ]:
# ============================================================
# HÜCRE 12: Hata konsolidasyonu
# ============================================================
HATA_SINIFI_ETIKETLERI = {
    "_hata_bos": ("Boş kayıt", "OTOMATİK_SİL"),
    "_hata_sembol": ("Sadece sayı/sembol", "OTOMATİK_SİL"),
    "_hata_tam_tekrar": ("Tam metin tekrarı", "OTOMATİK_SİL"),
    "_hata_benzer_guclu": ("Çok yüksek benzerlik", "GÜÇLÜ_ADAY"),
    "_hata_benzer_incele": ("Yüksek benzerlik", "İNCELE"),
    "_hata_kisa": ("Çok kısa metin", "İNCELE"),
    "_hata_uzun": ("Çok uzun metin", "İNCELE"),
    "_hata_dusuk_ttr": ("Düşük bilgi yoğunluğu", "İNCELE"),
    "_hata_dil": ("Türkçe dışı içerik", "İNCELE"),
    "_hata_yazim": ("Yazım hatası yoğun", "İNCELE"),
    "_hata_outlier": ("Konu dışı/aykırı", "İNCELE"),
    "_hata_kategori": ("Kategori uyumsuz", "İNCELE"),
}

AKSIYON_ONCELIGI = {"OTOMATİK_SİL": 4, "GÜÇLÜ_ADAY": 3, "İNCELE": 2, "TEMİZ": 0}
veri["_hata_siniflari"] = ""
veri["_oneri_aksiyon"] = "TEMİZ"

for idx in veri.index:
    siniflar = []
    en_kotu = "TEMİZ"
    for kolon, (etiket, aksiyon) in HATA_SINIFI_ETIKETLERI.items():
        if kolon in veri.columns and veri.at[idx, kolon]:
            siniflar.append(etiket)
            if AKSIYON_ONCELIGI[aksiyon] > AKSIYON_ONCELIGI[en_kotu]:
                en_kotu = aksiyon
    veri.at[idx, "_hata_siniflari"] = "; ".join(siniflar) if siniflar else ""
    veri.at[idx, "_oneri_aksiyon"] = en_kotu

print("=" * 50)
print("📊 KONSOLİDASYON")
print("=" * 50)
for kolon, (etiket, _) in HATA_SINIFI_ETIKETLERI.items():
    s = veri[kolon].sum() if kolon in veri.columns else 0
    if s > 0: print(f"  {etiket}: {s}")

aksiyon_dagilimi = veri["_oneri_aksiyon"].value_counts()
print(f"\n📊 Aksiyon:")
for a, s in aksiyon_dagilimi.items(): print(f"  {a}: {s}")

temiz_satir = (veri["_oneri_aksiyon"] == "TEMİZ").sum()
hatali_satir = len(veri) - temiz_satir
print(f"\n📊 {len(veri)} toplam | {hatali_satir} hatalı/şüpheli (%{hatali_satir/len(veri)*100:.1f}) | {temiz_satir} temiz")
print("✅ Hücre 12 tamam.")

## 🎯 Hücre 13: Eşik Kalibrasyonu (Doğrulama Seti ile)

In [ ]:
# ============================================================
# HÜCRE 13: Kalibrasyon
# ============================================================
dogrulama_dosyasi = os.path.join(CIKTI_DIZINI, "dogrulama_seti.xlsx")
if os.path.exists(dogrulama_dosyasi):
    etiket_verisi = pd.read_excel(dogrulama_dosyasi, engine="openpyxl")
    etiketli = etiket_verisi[etiket_verisi["_manuel_etiket"].isin(["temiz", "hatali", "sil"])]
    if len(etiketli) > 10:
        dogrulama_sonuc = veri.loc[dogrulama_indeksleri[:len(etiket_verisi)]].copy()
        dogrulama_sonuc["_manuel_etiket"] = etiket_verisi["_manuel_etiket"].values[:len(dogrulama_sonuc)]
        dogrulama_sonuc = dogrulama_sonuc[dogrulama_sonuc["_manuel_etiket"].isin(["temiz", "hatali", "sil"])]
        if len(dogrulama_sonuc) > 10:
            from sklearn.metrics import classification_report, confusion_matrix
            y_true = dogrulama_sonuc["_manuel_etiket"].apply(lambda x: 1 if x in ["sil", "hatali"] else 0)
            y_pred = dogrulama_sonuc["_oneri_aksiyon"].apply(lambda x: 0 if x == "TEMİZ" else 1)
            print("=" * 50)
            print("📊 DOĞRULAMA SETİ SONUÇLARI")
            print("=" * 50)
            print(f"Etiketli: {len(dogrulama_sonuc)}")
            print(f"\nConfusion Matrix:\n{confusion_matrix(y_true, y_pred, labels=[0, 1])}")
            print(f"\n{classification_report(y_true, y_pred, target_names=['TEMİZ', 'HATALI'])}")
            yp_samples = dogrulama_sonuc[(y_true == 0) & (y_pred == 1)]
            yn_samples = dogrulama_sonuc[(y_true == 1) & (y_pred == 0)]
            if len(yp_samples) > 0:
                print(f"\n⚠️ Yanlış pozitif (temiz→hatalı): {len(yp_samples)}")
            if len(yn_samples) > 0:
                print(f"⚠️ Yanlış negatif (hatalı→temiz): {len(yn_samples)}")
        else:
            print("⚠️ Yeterli etiketli satır yok.")
    else:
        print("⚠️ Etiketlenmiş satır bulunamadı.")
else:
    print("⚠️ 'dogrulama_seti.xlsx' bulunamadı. Hücre 3'te oluşturun.")
print("✅ Hücre 13 tamam.")

## 📤 Hücre 14: Çıktıları Üret

**v2.1:** 3 çıktı dosyası:
1. `korunan_veri.xlsx` — OTOMATİK_SİL olmayan TÜM satırlar (İNCELE + GÜÇLÜ_ADAY + TEMİZ)
2. `otomatik_silinecekler.xlsx` — Sadece OTOMATİK_SİL satırlar
3. `tum_kayitlar_raporu.xlsx` — Tüm satırlar + hata bilgileri

In [ ]:
# ============================================================
# HÜCRE 14: Çıktı dosyaları
# ============================================================

orijinal_kolonlar = [k for k in veri.columns if not k.startswith("_") and "embed" not in k.lower()]

# --- 1. Korunan veri (OTOMATİK_SİL olmayan her şey) ---
korunan_veri = veri[veri["_oneri_aksiyon"] != "OTOMATİK_SİL"].copy()
korunan_kolonlar = [k for k in orijinal_kolonlar if k in korunan_veri.columns]
korunan_yolu = os.path.join(CIKTI_DIZINI, "korunan_veri.xlsx")
korunan_veri[korunan_kolonlar].to_excel(korunan_yolu, index=False, engine="openpyxl")
print(f"✅ Korunan veri: {korunan_yolu} ({len(korunan_veri)} satır — İNCELE + GÜÇLÜ_ADAY + TEMİZ)")

# --- 2. Sadece OTOMATİK_SİL ---
silinecek_veri = veri[veri["_oneri_aksiyon"] == "OTOMATİK_SİL"].copy()
silinecek_yolu = os.path.join(CIKTI_DIZINI, "otomatik_silinecekler.xlsx")
silinecek_veri[korunan_kolonlar].to_excel(silinecek_yolu, index=False, engine="openpyxl")
print(f"✅ Otomatik silinecek: {silinecek_yolu} ({len(silinecek_veri)} satır)")

# --- 3. Tam rapor ---
rapor_kolonlari = [k for k in orijinal_kolonlar if k in veri.columns]
ek_kolonlar = ["_hata_siniflari", "_oneri_aksiyon", "_benzer_seviye", "_benzer_skor", "_benzer_madde_no", "_benzer_metin",
               "_dil", "_yazim_orani", "_token_sayisi", "_ttr", "_kategori_cosine"]
ek_kolonlar = [k for k in ek_kolonlar if k in veri.columns]
rapor_veri = veri[rapor_kolonlari + ek_kolonlar].copy()
rapor_veri.columns = [c.replace("_", " ").strip().title() for c in rapor_veri.columns]
rapor_yolu = os.path.join(CIKTI_DIZINI, "tum_kayitlar_raporu.xlsx")
rapor_veri.to_excel(rapor_yolu, index=False, engine="openpyxl")
print(f"✅ Tam rapor: {rapor_yolu} ({len(rapor_veri)} satır)")

# --- JSON özet ---
ozet = {"olusturma_tarihi": datetime.now().isoformat(), "kaynak_dosya": os.path.basename(EXCEL_DOSYASI),
         "toplam_satir": int(len(veri)), "temiz_satir": int(temiz_satir), "hatali_satir": int(hatali_satir),
         "aksiyon_dagilimi": {k: int(v) for k, v in aksiyon_dagilimi.items()},
         "hata_dagilimi": {}, "model": "paraphrase-multilingual-mpnet-base-v2", "boyut": int(boyut)}
for kolon, (etiket, _) in HATA_SINIFI_ETIKETLERI.items():
    s = int(veri[kolon].sum()) if kolon in veri.columns else 0
    if s > 0: ozet["hata_dagilimi"][etiket] = s
with open(os.path.join(CIKTI_DIZINI, "ozet.json"), "w", encoding="utf-8") as f:
    json.dump(ozet, f, ensure_ascii=False, indent=2)

print(f"\n{'='*50}")
print(f"📁 {CIKTI_DIZINI}/")
print(f"   📄 korunan_veri.xlsx           → {len(korunan_veri)} satır (hepsi korunur)")
print(f"   📄 otomatik_silinecekler.xlsx   → {len(silinecek_veri)} satır (sadece OTOMATİK_SİL)")
print(f"   📄 tum_kayitlar_raporu.xlsx     → {len(rapor_veri)} satır")
print(f"   📄 ozet.json")
print(f"{'='*50}")
print("🎉 İŞLEM TAMAMLANDI!")
print("\n📌 Sonraki adım:")
print("   1. 'otomatik_silinecekler.xlsx' → gözden geçir")
print("   2. 'korunan_veri.xlsx' → HadesBot'a aktar")
print("      python excel_ice_aktar.py ciktilar/korunan_veri.xlsx")
print("      python vektorleri_olustur.py")

---
## 📝 v2.1 Değişiklik Notları (v2'den)

- 🔴 `score_function="cos_sim"` → `from sentence_transformers.util import dot_score` + `score_function=dot_score`
- 🔴 Tam tekrar `set` mantığı kaldırıldı → doğrudan normalize string karşılaştırma
- 🔴 Doğrulama seti `if not os.path.exists()` koruması
- 🟠 Madde no embedding metninden çıkarıldı
- 🟠 Leave-one-out centroid (kayıt kendini centroid'den çıkarır)
- 🟠 Rastgele örneklem (`sample(random_state=42)`)
- 🟠 2 çıktı: `korunan_veri.xlsx` + `otomatik_silinecekler.xlsx`
- 🟢 FAISS açıklaması düzeltildi → "parçalı cosine hesaplanır"
- 🟢 `torch` pip listesinden çıkarıldı
- 🟢 `COLAB_MI` ortam kontrolü

---
**HadesBot · UYDS SSS Veri Ayıklama v2.1 · 2026**